In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from _load_dataset import load_dataset_sparse_labels

In [2]:
# inclusive range of seeds
SEED_RANGE = (0, 1000)

# choose to minimize or maximize MSE
DIRECTION = "maximize"  # "minimize" or "maximize"

In [3]:
# load data
_, s008_lidar, _, _, s009_lidar, _ = load_dataset_sparse_labels()

# prepare output folder
out_dir = "runs/pca_seed"
os.makedirs(out_dir, exist_ok=True)

/home/matheus/src/RayWise/src/_load_dataset.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)
/home/matheus/src/RayWise/src/_load_dataset.py:65: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input shape: (9234, 2)
lidar_input shape: (9234, 20, 200, 10)
Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)
y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)
Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)


/home/matheus/src/RayWise/src/_load_dataset.py:101: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


In [4]:
s008_flat = s008_lidar.reshape(s008_lidar.shape[0], -1)
s009_flat = s009_lidar.reshape(s009_lidar.shape[0], -1)

In [ ]:
results = []
for seed in range(SEED_RANGE[0], SEED_RANGE[1] + 1):
    # split data
    data008, _ = train_test_split(s008_flat, train_size=0.8, random_state=seed)
    _, data009 = train_test_split(s009_flat, test_size=0.2, random_state=seed)

    # fit PCA
    pca8 = PCA(n_components=3, random_state=seed)
    pca9 = PCA(n_components=3, random_state=seed)
    pca8.fit(data008)
    pca9.fit(data009)

    # compute explained variance difference
    ev8 = pca8.explained_variance_ratio_
    ev9 = pca9.explained_variance_ratio_
    diff = ev8 - ev9

    # compute metrics
    l1 = np.sum(np.abs(diff))
    l2 = np.sqrt(np.sum(diff**2))
    linf = np.max(np.abs(diff))
    mse = np.mean(diff**2)
    results.append((seed, l1, l2, linf, mse))

    # project data
    proj008 = pca8.transform(data008)
    proj009 = pca9.transform(data009)

    # plot 3D scatter
    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(proj008[:, 0], proj008[:, 1], proj008[:, 2], label="s008", alpha=0.5)
    ax.scatter(proj009[:, 0], proj009[:, 1], proj009[:, 2], label="s009", alpha=0.5)
    ax.set_title(f"Seed {seed}  " f"L1={l1:.3f}  L2={l2:.3f}  Linf={linf:.3f}  MSE={mse:.6f}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_zlabel("PC3")
    ax.legend()

    # save plot
    fname = f"seed_{seed}_" f"L1_{l1:.3f}_" f"L2_{l2:.3f}_" f"Linf_{linf:.3f}_" f"MSE_{mse:.6f}.png"
    fig.savefig(os.path.join(out_dir, fname))
    plt.close(fig)
    
    print (f"Processed seed {seed}: L1={l1:.3f}, L2={l2:.3f}, Linf={linf:.3f}, MSE={mse:.6f}")

Processed seed 0: L1=0.126, L2=0.115, Linf=0.115, MSE=0.004416
Processed seed 1: L1=0.127, L2=0.114, Linf=0.114, MSE=0.004362
Processed seed 2: L1=0.132, L2=0.119, Linf=0.118, MSE=0.004719
Processed seed 3: L1=0.133, L2=0.119, Linf=0.119, MSE=0.004747
Processed seed 4: L1=0.144, L2=0.125, Linf=0.124, MSE=0.005212
Processed seed 5: L1=0.140, L2=0.121, Linf=0.121, MSE=0.004907
Processed seed 6: L1=0.137, L2=0.122, Linf=0.122, MSE=0.005001
Processed seed 7: L1=0.136, L2=0.122, Linf=0.121, MSE=0.004946
Processed seed 8: L1=0.142, L2=0.124, Linf=0.124, MSE=0.005154
Processed seed 9: L1=0.144, L2=0.126, Linf=0.125, MSE=0.005258
Processed seed 10: L1=0.139, L2=0.121, Linf=0.120, MSE=0.004887
Processed seed 11: L1=0.141, L2=0.125, Linf=0.125, MSE=0.005240
Processed seed 12: L1=0.132, L2=0.117, Linf=0.116, MSE=0.004525
Processed seed 13: L1=0.137, L2=0.119, Linf=0.119, MSE=0.004754
Processed seed 14: L1=0.152, L2=0.130, Linf=0.129, MSE=0.005664
Processed seed 15: L1=0.144, L2=0.127, Linf=0.127,

In [ ]:
# print metrics table
print("L1 -> L1 norm, this sums the absolute per-component differences.")
print("L2 -> L2 norm, this is the Euclidean distance between the two vectors.")
print("Linf -> Linf norm, this is the maximum absolute per-component difference.")
print("MSE -> Mean Squared Error, this is the average of the squared differences.\n")

print("seed   L1       L2       Linf     MSE")
for seed, l1, l2, linf, mse in results:
    print(f"{seed:>4} {l1:>8.3f} {l2:>8.3f} {linf:>8.3f} {mse:>10.6f}")

# select best seed
if DIRECTION == "minimize":
    best = min(results, key=lambda x: x[4])
elif DIRECTION == "maximize":
    best = max(results, key=lambda x: x[4])
else:
    raise ValueError('DIRECTION must be "minimize" or "maximize"')

best_seed, _, _, _, best_mse = best
print(f"Best seed for {DIRECTION} MSE: {best_seed} (MSE={best_mse:.6f})")